# Etapa 1 — EDA, métricas e baselines (Telco Churn)

**Entregável:** notebook de EDA + baselines registrados no MLflow.

**Referências do material FIAP (transcrições em `disciplinas/`):**
- *Ciclo de Vida — Aula 01:* entendimento de negócio, KPIs, variáveis, stakeholders (`aula_01_entendendo-o-problema-de-negocio-e-os-dados.txt`).
- *Ciclo de Vida — Aula 02:* EDA, MVP, `pandas`/`sklearn`, inspeção inicial (`aula_02_experimentacao-e-mvp-de-modelos.txt`).
- *Fundamentos — Aula 01:* aprendizado supervisionado, `train_test_split`, avaliação (`aula_01_machine-learning-aula-01.txt`).
- *Fundamentos — Aula 02:* pipelines com `ColumnTransformer`, `OneHotEncoder`, `StandardScaler` (`aula_02_machine-learning-aula-02.txt`).
- *Fundamentos — Aula 06 (métricas / ROC-PR):* precisão, revocação, F1, curvas ROC e PR (`aula_06_machine-learning-aula-06.txt`). O roteiro do challenge cita “Aula 05” para métricas; neste repositório de extratos, a discussão detalhada de ROC/PR está na Aula 06.

O **ML Canvas** preenchido está em `docs/ml-canvas.md`.

In [ ]:
import hashlib
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


def project_root() -> Path:
    """Raiz do repo (funciona com cwd em `notebooks/` ou na raiz)."""
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return cwd


ROOT = project_root()
DATA_PATH = ROOT / "data" / "raw" / "Telco-Customer-Churn.csv"
MLRUNS = ROOT / "mlruns"

sns.set_theme(style="whitegrid", context="notebook")
print("ROOT:", ROOT)

## 1. Carga dos dados

Execute antes `python scripts/download_data.py` (ou rode a célula abaixo uma vez). Fonte: repositório público IBM Telco Customer Churn.

In [ ]:
# Baixa CSV se ainda não existir
import subprocess
import sys

if not DATA_PATH.exists():
    subprocess.run([sys.executable, str(ROOT / "scripts" / "download_data.py")], check=True)

df = pd.read_csv(DATA_PATH)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(df.shape)
df.head()

## 2. Data readiness e qualidade (volume, tipos, faltantes)

Espelha o fluxo da *Aula 02* (Ciclo de Vida): `info`, `describe`, missing values, target.

In [ ]:
print(df.info())
print("\n--- Missing por coluna (%) ---")
miss = df.isna().mean().mul(100).sort_values(ascending=False)
print(miss[miss > 0].round(2).to_string() if miss.max() > 0 else "Sem NaN")

df["Churn"] = (df["Churn"] == "Yes").astype(int)
if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

print("\nDistribuição do alvo (Churn):")
print(df["Churn"].value_counts(normalize=True).round(3))
print("\n--- describe() (numéricas) ---")
display(df.describe(include=[np.number]).T)
print("\n--- describe() (categóricas, amostra) ---")
display(df.describe(include="object").T.head(20))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(x="Churn", data=df, ax=axes[0])
axes[0].set_title("Alvo (0=mantém, 1=churn)")
num_cols = [c for c in ["tenure", "MonthlyCharges", "TotalCharges"] if c in df.columns]
df[num_cols].plot.hist(bins=30, alpha=0.5, ax=axes[1])
axes[1].set_title("Distribuição (numéricas)")
axes[1].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 8))
corr = df.select_dtypes(include=[np.number]).corr(numeric_only=True)
sns.heatmap(corr, annot=False, cmap="vlag", center=0)
plt.title("Correlação (numéricas)")
plt.tight_layout()
plt.show()

## 3. Métricas técnicas e de negócio

**Técnicas (challenge + Fundamentos):** ROC-AUC, PR-AUC (`average_precision`), F1 no limiar 0,5 (para comparar baselines de forma simples).

**Negócio:** *custo de churn evitado* como cenário ilustrativo — parâmetros `LTV_medio` e `custo_acao` ajustáveis com stakeholders (ver também `docs/ml-canvas.md`).

In [ ]:
LTV_MEDIO = 1200.0  # receita vitalícia média fictícia (R$)
CUSTO_ACAO = 50.0   # custo de contato/oferta fictício (R$)


def business_value_proxy(y_true, y_pred, ltov=LTV_MEDIO, cost=CUSTO_ACAO):
    """Valor esperado simples: VP evita churn (ltov), FP paga custo de ação."""
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    return float(tp * ltov - fp * cost)


def eval_classifiers(name, pipe, X, y, cv):
    scoring = {
        "roc_auc": "roc_auc",
        "avg_prec": make_scorer(average_precision_score, needs_proba=True),
    }
    res = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=1,
    )
    # F1 e proxy de negócio no fold: precisamos de y_pred por fold
    f1_folds = []
    biz_folds = []
    for train_idx, test_idx in cv.split(X, y):
        Xt, Xv, yt, yv = X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]
        pipe.fit(Xt, yt)
        proba = pipe.predict_proba(Xv)[:, 1]
        pred = (proba >= 0.5).astype(int)
        f1_folds.append(f1_score(yv, pred))
        biz_folds.append(business_value_proxy(yv, pred))
    metrics = {
        "roc_auc_mean": float(np.mean(res["test_roc_auc"])),
        "roc_auc_std": float(np.std(res["test_roc_auc"])),
        "pr_auc_mean": float(np.mean(res["test_avg_prec"])),
        "pr_auc_std": float(np.std(res["test_avg_prec"])),
        "f1_mean": float(np.mean(f1_folds)),
        "f1_std": float(np.std(f1_folds)),
        "business_proxy_mean": float(np.mean(biz_folds)),
        "business_proxy_std": float(np.std(biz_folds)),
    }
    return metrics

## 4. Pré-processamento e baselines (*Fundamentos* Aulas 01–02)

- `DummyClassifier` (estratificado — respeita proporção do alvo).
- `LogisticRegression` com pipeline numérico + categórico (mesma ideia de `ColumnTransformer` da Aula 02 de regressão regularizada, adaptada à classificação).

In [ ]:
feature_cols = [c for c in df.columns if c != "Churn"]
X = df[feature_cols].copy()
y = df["Churn"].copy()

num_features = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_features = [c for c in feature_cols if c not in num_features]

numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_features),
        ("cat", categorical_pipe, cat_features),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

dummy = Pipeline(
    steps=[
        ("prep", preprocess),
        ("model", DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)),
    ]
)

logit = Pipeline(
    steps=[
        ("prep", preprocess),
        (
            "model",
            LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, class_weight="balanced"),
        ),
    ],
)

results = []
for label, model in [("dummy_stratified", dummy), ("logistic_regression_balanced", logit)]:
    m = eval_classifiers(label, model, X, y, cv)
    m["model"] = label
    results.append(m)

pd.DataFrame(results).set_index("model").round(4)

## 5. MLflow — experimentos (*Ciclo de Vida* Aula 02 + menções em Aulas 06–08)

Registramos parâmetros, métricas agregadas na CV e fingerprint do arquivo de dados.

In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


dataset_sha = sha256_file(DATA_PATH) if DATA_PATH.exists() else "missing"

mlflow.set_tracking_uri(f"file:{MLRUNS}")
mlflow.set_experiment("etapa1-telco-churn-baselines")

common_params = {
    "random_state": RANDOM_STATE,
    "cv_splits": cv.n_splits,
    "dataset_path": str(DATA_PATH.relative_to(ROOT)) if DATA_PATH.is_relative_to(ROOT) else str(DATA_PATH),
    "dataset_sha256": dataset_sha,
    "n_rows": len(df),
    "n_features_raw": len(feature_cols),
    "LTV_MEDIO_hypothesis": LTV_MEDIO,
    "CUSTO_ACAO_hypothesis": CUSTO_ACAO,
}

for label, model in [("dummy_stratified", dummy), ("logistic_regression_balanced", logit)]:
    metrics = eval_classifiers(label, model, X, y, cv)
    with mlflow.start_run(run_name=label):
        mlflow.log_params({**common_params, "model_name": label})
        for k, v in metrics.items():
            mlflow.log_metric(k, v)
print("Runs gravados em", MLRUNS)
print("UI: mlflow ui --backend-store-uri file:%s" % MLRUNS)

## 6. Próximos passos (Etapa 2)

- MLP em PyTorch, early stopping, comparação com estes baselines e novas métricas nos mesmos folds ou protocolo fixo.

## 7. Etapa 2 — Comparação MLP vs baselines (tarefa 3)

Holdout **estratificado** (mesmo split para todos): `DummyClassifier` estratificado, **regressão logística** (linear), **Random Forest** e **HistGradientBoosting** (árvores / ensemble em árvores), além da **MLP** (`telco_churn`).

Métricas no conjunto de validação (≥4): **ROC-AUC**, **PR-AUC** (`average_precision`), **F1** e **acurácia** (limiar 0,5 nas probabilidades).

Requer `pip install -e .` na raiz do repo para importar o pacote `telco_churn`.

In [ ]:
import sys

ROOT = project_root()
sys.path.insert(0, str(ROOT / "src"))

from telco_churn import (
    TrainConfig,
    compare_models_holdout,
    prepare_telco_features,
)

X_cmp, y_cmp = prepare_telco_features(df)

comparison, val_artifacts = compare_models_holdout(
    X_cmp,
    y_cmp,
    random_state=RANDOM_STATE,
    test_size=0.2,
    mlp_hidden_dims=(64, 32),
    mlp_train_config=TrainConfig(
        batch_size=64,
        max_epochs=300,
        patience=25,
        learning_rate=1e-3,
        seed=RANDOM_STATE,
    ),
    return_val_artifacts=True,
)
comparison.round(4)

## 8. Etapa 2 — Trade-off de custo FP vs FN (tarefa 4)

Premissas ilustrativas (alinhadas ao notebook / ML Canvas): **custo por falso positivo** (contato em cliente que não churnaria) e **custo por falso negativo** (churn não detectado, proxy de LTV não retida).

No mesmo **conjunto de validação** da comparação de modelos (`val_artifacts`): tabela **limiar 0,5** vs **limiar que minimiza** `FP × c_fp + FN × c_fn`, e curva de **FP** e **FN** em função do limiar (tensão operacional entre alertas indevidos e churns perdidos).

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from telco_churn import (
    DEFAULT_COST_FN,
    DEFAULT_COST_FP,
    compare_thresholds_report,
    sweep_threshold_costs,
)

y_val = val_artifacts["y_val"]
proba_mlp = val_artifacts["scores"]["churn_mlp"]

rep = compare_thresholds_report(
    y_val,
    proba_mlp,
    baseline_threshold=0.5,
    cost_fp=DEFAULT_COST_FP,
    cost_fn=DEFAULT_COST_FN,
)
display(rep.round(2))

cost_curve = sweep_threshold_costs(
    y_val,
    proba_mlp,
    cost_fp=DEFAULT_COST_FP,
    cost_fn=DEFAULT_COST_FN,
)
ax = cost_curve.plot(x="threshold", y=["fp", "fn"], figsize=(9, 4), title="FP vs FN por limiar (MLP)")
ax.set_xlabel("threshold (P(churn) ≥ t → intervenção)")
plt.tight_layout()
plt.show()


## 9. Etapa 2 — MLflow (tarefa 5)

Um **run por modelo** (dummy, logística, RF, HGB, MLP) com parâmetros de protocolo, **SHA-256 do CSV** (quando o arquivo existe), métricas de holdout e hiperparâmetros da MLP.

**Artefatos (padrão do `log_compare_models_to_mlflow`):** curva de loss da MLP em CSV (`training/`), **pipelines sklearn** (`sklearn_model/`) e **MLP PyTorch** (`torch_mlp/`). Para runs mais leves (ex.: CI), use `log_sklearn_models=False` e `log_mlp_torch=False`.

In [ ]:
import hashlib
from pathlib import Path

from telco_churn import TrainConfig, log_compare_models_to_mlflow


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


ROOT = project_root()
MLRUNS_2 = ROOT / "mlruns"
DATA_PATH_ML = ROOT / "data" / "raw" / "Telco-Customer-Churn.csv"
dataset_sha_ml = sha256_file(DATA_PATH_ML) if DATA_PATH_ML.exists() else "missing"

mlflow_table = log_compare_models_to_mlflow(
    X_cmp,
    y_cmp,
    tracking_uri=f"file:{MLRUNS_2}",
    experiment_name="etapa2-telco-churn-compare",
    dataset_sha256=dataset_sha_ml,
    extra_params={"notebook": "01_eda_baselines", "random_state_notebook": RANDOM_STATE},
    run_name_prefix="etapa2-",
    log_training_curves=True,
    # log_sklearn_models / log_mlp_torch: True por padrão (modelos no MLflow)
    random_state=RANDOM_STATE,
    test_size=0.2,
    mlp_hidden_dims=(64, 32),
    mlp_train_config=TrainConfig(
        batch_size=64,
        max_epochs=300,
        patience=25,
        learning_rate=1e-3,
        seed=RANDOM_STATE,
    ),
)
mlflow_table.round(4)
